In [16]:
model = "llama3.2:1B"

#### Task 1: Create a Simple Chain for Summarization


**Objective:**

Build a LangChain chain that can summarize a given text.

**Task Description:**

- Create a llm chain using with a Ollama model.
- Define a prompt template for summarization. The summary should be only one sentence.
- Run the chain with a sample text and print the summary.
- Add model output streaming.
- Run the chain with streaming with a sample text and print the summary.


**Hint: The five messages in LangChain:**


- SystemMessage: corresponds to system role
- HumanMessage: corresponds to user role
- AIMessage: corresponds to assistant role
- AIMessageChunk: corresponds to assistant role, used for streaming responses
- ToolMessage: corresponds to tool role


[More Information](https://python.langchain.com/docs/concepts/messages/)

**Useful links:**

- [How To Prompt Template 1](https://python.langchain.com/v0.2/docs/tutorials/extraction/#the-extractor)
- [How To Prompt Template 2](https://python.langchain.com/v0.2/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
- [How To LCEL Chains 1](https://python.langchain.com/v0.2/docs/concepts/#langchain-expression-language-lcel)
- [How To LCEL Chains 2](https://python.langchain.com/v0.2/docs/versions/migrating_chains/llm_chain/#lcel)
- [How To Chain Streaming](https://python.langchain.com/v0.2/docs/concepts/#streaming)

In [17]:
!python -m ensurepip --default-pip
!python -m pip install --upgrade pip
!pip install langchain openai chromadb tiktoken


Looking in links: c:\Users\Dominik\AppData\Local\Temp\tmpssa3ny2x


In [18]:
import sys
!{sys.executable} -m pip install langchain-ollama


In [19]:
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# Load the Ollama model
llm = ChatOllama(model="llama3.2:1b")

# ADD HERE YOUR CODE
# Define the prompt template

prompt1 = ChatPromptTemplate.from_template(
"Write a concise summary (no more than 40 words) of the following paragraph. Focus on the definition of Generative AI and its main fields of application. Use clear academic language, and avoid examples or unnecessary adjectives. Paragraph: Generative AI is a field of artificial intelligence focused on creating new content based on patterns learned from existing data. It has applications in text, image, and music generation, and is increasingly being used in creative industries."
)

# ADD HERE YOUR CODE
# Create the LLMChain

summarization_chain = prompt1 | llm 

# Sample text
text = """Over the last decade, deep learning has evolved massively to process and generate unstructured data like text, images, and video. 
These advanced AI models have gained popularity in various industries, and include large language models (LLMs). 
There is currently a significant level of fanfare in both the media and the industry surrounding AI,
and there’s a fair case to be made that Artificial Intelligence (AI), with these advancements,
is about to have a wide-ranging and major impact on businesses, societies, and individuals alike.
This is driven by numerous factors, including advancements in technology, high-profile applications, 
and the potential for transfor- mative impacts across multiple sectors."""

# ADD HERE YOUR CODE
# Invoke the chain
summary = summarization_chain.invoke({"text": text})
print(summary.content)

Generative AI refers to the development of algorithms that can generate novel or synthetic data based on statistical patterns observed in existing datasets. This field has various applications, including text and image generation, as well as music composition, with increasing use in creative industries such as film and literature.


In [20]:
# Stream the chain output
for chunk in summarization_chain.stream({"text": text}):
    print(chunk.content, end="", flush=True)

Generative AI refers to artificial intelligence that creates novel content by analyzing patterns in existing data. This field applies to the development of models for generating text, images, and music, with increasing use in fields such as creative industries, where AI-driven content creation can enhance artistic expression and product innovation.

#### Task 2: Chain with Tool Usage (Simple Math Tool)

**Objective:**

Create a LangChain chain that uses a simple math tool to perform calculations.

**Task Description:**

- Define a function as tool which multiplies two integer values and return the result.
- Create a chain
- Print the result of the calculation.

**Useful links:**

- [How To Tools](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#create-a-tool)
- [How To Tools in Chains](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#chains)
- [How To Tool Calling](https://python.langchain.com/v0.2/docs/concepts/#functiontool-calling)
- [How To Chain and Call Tools with Ollama](https://python.langchain.com/v0.2/docs/integrations/chat/ollama/)

In [24]:
from langchain_core.tools import tool


# ADD HERE YOUR CODE
# Create custom tool
@tool


def multiply(first_int: int, second_int: int) -> int:
    """Multiply two integers and return the result."""
    return first_int * second_int

##print(multiply.name)
##print(multiply.description)
##print(multiply.args) # -> definition of tool arguments

# Invoke custom tool
result = multiply.invoke({"first_int": 4, "second_int": 5})
print(result)

20


In [ ]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

# Load the Ollama model
llm = ChatOllama(model="llama3.2:1b")

# ADD HERE YOUR CODE
# Use bind_tools to pass the definition of our tool in as part of each call to the model, so that the model can invoke the tool
llm_with_tools = llm.bind_tools([multiply])

# When the model invokes the tool, this will show up in the AIMessage.tool_calls attribute of the output -> extract tool parameters from input text
msg = llm_with_tools.invoke("whats 5 times forty two")
print(msg.tool_calls)



[{'name': 'multiply', 'args': {'first_int': '42', 'second_int': '5'}, 'id': '4ac9f567-988d-4bca-aae1-7f37b436a69a', 'type': 'tool_call'}]


In [27]:
# ADD HERE YOUR CODE
# Create the chain: pass the extracte tool parameters from the input text to the tool -> extract the arguments of the first tool_call
from langchain_core.runnables import RunnableLambda

# Function to execute the tool call
def execute_first_tool_call(msg):
    tool_call = msg.tool_calls[0]
    args = tool_call["args"]
    return multiply.invoke(args)

# Make runnable
tool_executor = RunnableLambda(execute_first_tool_call)

# Build final chain
chain_with_tools = llm_with_tools | tool_executor

# Test chain
final_result = chain_with_tools.invoke("whats 5 times forty two")
print(final_result)  # => 210


210


#### Task 3: Agent with Tool Usage (Two Tools)

**Objective:** 

Create a LangChain agent that uses two tools to perform tasks.

**Task Description:**

- Define prompt template.
- Define tools.
- Create an Agent using the Ollama model, prompt template and tools.
- Run the agent with a prompt that requires one or both tools.
- Observe how the agent uses the tools to complete the task.


**Useful links:**

- [How To 1](https://python.langchain.com/v0.2/docs/concepts/#agents)
- [How To 2](https://python.langchain.com/v0.2/docs/how_to/tools_chain/#agents)
- [How To 3](https://python.langchain.com/v0.2/docs/tutorials/agents/)

In [37]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

model = "llama3.2:1b"

# Load the Ollama model
llm = ChatOllama(model=model)

# ADD HERE YOUR CODE
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise and accurate. Use the tools for math operations."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad")
])

# Custom math tools
@tool
def add(first_int: int, second_int: int) -> int:
    "Add two integers."
    return first_int + second_int


@tool
def exponentiate(base: int, exponent: int) -> int:
    "Exponentiate the base to the exponent power."
    return base**exponent


tools = [add, exponentiate]

In [38]:
# ADD HERE YOUR CODE
# Construct the tool calling agent
agent_with_tools = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=agent_prompt,
)

# ADD HERE YOUR CODE
agent_executor_with_tools = AgentExecutor(
    agent=agent_with_tools,
    tools=tools,
    verbose=True,   # zeigt dir im Output, wie die Tools aufgerufen werden
)

In [41]:
agent_executor_with_tools.invoke(
    {
        "input": "First take 3 to the power of five and afterwards add 12."
    }
)



> Entering new AgentExecutor chain...

Invoking: `add` with `{'first_int': '3', 'properties': "{'first_int': 'exponentiate', 'second_int': 'add'}", 'second_int': '5'}`


8The result of 3 to the power of 5 is 243, and adding 12 gives us:

243 + 12 = 255

> Finished chain.


{'input': 'First take 3 to the power of five and afterwards add 12.',
 'output': 'The result of 3 to the power of 5 is 243, and adding 12 gives us:\n\n243 + 12 = 255'}

#### [Optional] Task 4: Enhance Agent with Memory

**Objective:**

Eenhance the agent from Task 3 with memory to improve its context awareness and ability to maintain state.

**Instructions:**

- Create a ConversationBufferMemory to store chat history.
- Modify the agent to use the memory to inform its responses.
- Run the agent with a series of prompts that require context or state to be maintained.
- Observe how the agent's responses improve with the addition of memory.

**Useful links:**

- [How To Memory 1](https://python.langchain.com/v0.2/api_reference/langchain/memory/langchain.memory.buffer.ConversationBufferMemory.html#langchain.memory.buffer.ConversationBufferMemory)
- [How To Memory 1](https://python.langchain.com/v0.2/docs/versions/migrating_chains/conversation_chain/#legacy)

In [19]:
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import MessagesPlaceholder

# Load the Ollama model
llm = ChatOllama(model=llama3.2:1b)

# Define memory object for conversation history
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="output")

# ADD HERE YOUR CODE
# Add history placeholder to prompt
agent_prompt_with_memory = ...

# ADD HERE YOUR CODE
# Construct the tool calling agent
agent_with_tools_and_memory = ...

# ADD HERE YOUR CODE
# Create an agent executor by passing in the agent and tools
agent_executor_with_tools_and_memory = ...

SyntaxError: invalid decimal literal (2674267836.py, line 5)

In [ ]:
question = "Take 3 to the fifth power then add that 12?"
ai_msg_1 = agent_executor_with_tools_and_memory.invoke({"user_input": question})
print(ai_msg_1["output"])

second_question = "Explain how you have calculated the result."
ai_msg_2 = agent_executor_with_tools_and_memory.invoke({"user_input": second_question})
print(ai_msg_2["output"])